In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce_gold;

In [0]:
from pyspark.sql.functions import col, count, sum, avg

# Read from your silver tables
df_users = spark.table("ecommerce_silver.users")
df_countries = spark.table("ecommerce_silver.countries")

# Create a gold-layer aggregate table (e.g., user metrics by country)
df_gold_user_summary = (
    df_users
    .groupBy("country")
    .agg(
        count("identifierHash").alias("total_users"),
        sum("socialNbFollowers").alias("total_followers"),
        sum("socialProductsLiked").alias("total_products_liked")
    )
    .orderBy(col("total_users").desc())
)

# Write to the gold layer path and register the table
df_gold_user_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .option("path", "abfss://landing@saecommercedataprod001.dfs.core.windows.net/gold/user_country_summary/") \
    .saveAsTable("ecommerce_gold.user_country_summary")

print("Successfully created ecommerce_gold.user_country_summary table!")

Successfully created ecommerce_gold.user_country_summary table!
